# EEG sleep foundation — train on a Colab GPU

Track 03 (Sleep Onset). This notebook rebuilds the project on Colab and trains
the submissions there:

1. **Runtime → Change runtime type → GPU** (T4 is enough), then **Runtime → Run all**.
2. The Muse data (~1.1 GB) is downloaded from NEMAR on the first run and cached
   in Google Drive, so later sessions only extract it.
3. Each finished build (`build/<label>/`, `dist/<label>.zip`, the log) is copied
   to Drive right away, so a disconnect does not lose finished runs.

Generated by `scripts/make_colab_notebook.py`; regenerate it after editing the
solver locally.

In [ ]:
!nvidia-smi -L
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Select a GPU runtime first."

## Settings

In [ ]:
from pathlib import Path

PROJECT = Path("/content/EEG_sleep_foundation")
COMPETITION_REPO = "https://github.com/neural-interfaces26/2026-competition.git"
COMPETITION_COMMIT = "a66bc23d1a0e686b895c093a642b45c74970758b"   # the harness version used locally

USE_DRIVE = True    # cache the data and save the results in Google Drive
DRIVE_DIR = Path("/content/drive/MyDrive/EEG_sleep_foundation")

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)

## Code: official harness + this project's files

In [ ]:
import subprocess
if not (PROJECT / "competition").exists():
    subprocess.run(["git", "clone", "-q", COMPETITION_REPO,
                    str(PROJECT / "competition")], check=True)
subprocess.run(["git", "-C", str(PROJECT / "competition"), "checkout", "-q",
                COMPETITION_COMMIT], check=True)
for sub in ("submission", "scripts", "build", "dist"):
    (PROJECT / sub).mkdir(parents=True, exist_ok=True)
%cd {PROJECT}

In [ ]:
%%writefile /content/EEG_sleep_foundation/submission/cnn_gru.py
"""Track 03 (Sleep Onset) submission: an ensemble of CNN + band powers -> GRU
models that each output a time-to-N2 distribution.

Each 5 s window goes through two encoders:
- a 1-D CNN on the raw signal, to learn waveform shapes (alpha/theta
  rhythms, slow eye movements on AF7/AF8, spindles, K-complexes);
- log band powers (delta, theta, alpha, sigma, beta, high/EMG), absolute and
  relative, the classic sleep-scoring markers.
Their features, plus the time elapsed since the recording started, feed a
GRU that keeps its state across the windows of a recording (streamed
evaluation: one window per ``predict`` call, ``reset_state`` per recording).

Per-sleeper adaptation (``adapt``): the CNN input is scaled by the
recording's own running RMS per channel, and the band powers are also given
relative to the recording's running mean, so each window is judged against
the same sleeper's history. Both statistics are causal expanding means over
the windows seen so far.

The head outputs a distribution over the time to the first N2 epoch, in
5 s bins (0, 5, ..., 595, 600 = capped). The ensemble averages the members'
distributions; ``p[0]`` is the hazard (probability that N2 starts within the
next window) and the prediction is the ``q``-quantile of the distribution,
with ``q`` tuned on held-out subjects.

Contract (see competition/tracks/sleep_onset/README.md):
- input  X: float tensor (B, C, T) on meta["device"], raw microvolts
- output  : (B,) float latencies in seconds to the first N2 epoch, in [0, 600]

This file must stay self-contained: only ``torch`` and packages from the
competition's requirements.txt may be imported, and no other local modules.
"""

import os
import time

import numpy as np
import torch
from torch import nn

from benchmark_utils.base_solver import CompetSolver

CAP_S = 600.0
WEIGHTS_FILE = "weights.pt"

BIN_S = 5.0
N_BINS = int(CAP_S / BIN_S) + 1          # 0, 5, ..., 600 s
CLIP_UV = 500.0
BANDS = ((0.5, 4.0), (4.0, 8.0), (8.0, 12.0), (12.0, 15.0), (15.0, 30.0),
         (30.0, 60.0))
# Metric ranges and severity weights (objective.py).
RANGE_EDGES = (0.0, 40.0, 90.0, 300.0, 600.0)
RANGE_WEIGHTS = (10.0, 5.0, 3.0, 1.0)

CNN_CHANNELS = (32, 64, 96, 128)
BP_DIM = 64
GRU_HIDDEN = 128
GRU_LAYERS = 2
DROPOUT = 0.2

# Training-time choices; inference reads them back from weights.pt.
N_SEEDS = int(os.environ.get("SLEEP_N_SEEDS", 5))
ADAPT = os.environ.get("SLEEP_ADAPT", "1") == "1"

EPOCHS = 30
PATIENCE = 8
LR = 1e-3
WEIGHT_DECAY = 1e-4
RECS_PER_BATCH = 16
CHUNK = 120                              # TBPTT length, in windows (10 min)
VAL_FRACTION = 0.15
LABEL_SIGMA = 1.5                        # soft-label width, in bins
L1_WEIGHT = 0.1
QUANTILES = (0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6)
SEED = 0


def preprocess(X):
    """Raw microvolts (..., C, T) -> per-window DC removed, artifacts clipped."""
    X = X - X.mean(dim=-1, keepdim=True)
    return X.clamp(-CLIP_UV, CLIP_UV)


def quantile_seconds(p, q):
    """Distributions (..., K) over the 5 s bins -> their q-quantile, seconds."""
    cdf = p.cumsum(-1).contiguous()
    k = torch.searchsorted(cdf, torch.full(cdf.shape[:-1] + (1,), float(q),
                                           device=cdf.device))
    return (k.clamp_max(N_BINS - 1).squeeze(-1) * BIN_S).float()


class SleepOnsetCNNGRU(nn.Module):
    """One member: (B, L, C, T) sequences -> logits (B, L, K) over bins."""

    def __init__(self, n_chans, n_times, sfreq, adapt=ADAPT):
        super().__init__()
        self.n_chans, self.n_times, self.sfreq = n_chans, n_times, sfreq
        self.window_s = n_times / sfreq
        self.use_adapt = bool(adapt)
        self.register_buffer("adapt", torch.tensor(self.use_adapt))
        self.register_buffer("chan_std", torch.ones(n_chans))
        n_bp = 2 * len(BANDS) * n_chans + n_chans
        self.register_buffer("bp_mean", torch.zeros(n_bp))
        self.register_buffer("bp_std", torch.ones(n_bp))
        self.register_buffer(
            "bin_values", torch.arange(N_BINS, dtype=torch.float32) * BIN_S)

        # Band masks over the rFFT bins, from the actual sampling rate.
        freqs = torch.fft.rfftfreq(n_times, d=1.0 / sfreq)
        nyq = sfreq / 2
        masks = torch.stack([((freqs >= lo) & (freqs < min(hi, nyq))).float()
                             for lo, hi in BANDS])
        self.register_buffer("band_masks", masks, persistent=False)
        self.register_buffer("hann", torch.hann_window(n_times),
                             persistent=False)

        layers, in_ch = [], n_chans
        kernels = (max(3, int(sfreq // 2)) | 1, 9, 7, 5)
        for out_ch, k in zip(CNN_CHANNELS, kernels):
            layers += [nn.Conv1d(in_ch, out_ch, k, padding=k // 2, bias=False),
                       nn.BatchNorm1d(out_ch), nn.GELU(), nn.MaxPool1d(2)]
            in_ch = out_ch
        self.cnn = nn.Sequential(*layers)
        bp_in = n_bp * (2 if self.use_adapt else 1)
        self.bp_proj = nn.Sequential(nn.Linear(bp_in, BP_DIM), nn.GELU())
        self.in_proj = nn.Sequential(
            nn.Linear(2 * CNN_CHANNELS[-1] + BP_DIM + 1, GRU_HIDDEN),
            nn.GELU(), nn.Dropout(DROPOUT))
        self.gru = nn.GRU(GRU_HIDDEN, GRU_HIDDEN, GRU_LAYERS, batch_first=True,
                          dropout=DROPOUT)
        self.head = nn.Sequential(nn.Dropout(DROPOUT),
                                  nn.Linear(GRU_HIDDEN, N_BINS))
        self.reset_state()

    def band_powers(self, X):
        """Preprocessed (N, C, T) -> raw log band-power features (N, F)."""
        spec = torch.fft.rfft(X * self.hann, dim=-1).abs() ** 2
        power = torch.einsum("nct,bt->ncb", spec, self.band_masks)
        log_abs = torch.log(power + 1e-6)
        log_rel = torch.log(power / (power.sum(-1, keepdim=True) + 1e-6)
                            + 1e-6)
        theta_alpha = log_abs[..., 1] - log_abs[..., 2]
        return torch.cat([log_abs.flatten(1), log_rel.flatten(1),
                          theta_alpha], dim=1)

    def encode(self, X, start_index, sums):
        """Sequences (B, L, C, T) -> GRU inputs (B, L, H), new running sums.

        ``sums`` = (sum of band powers (B, F), sum of channel variances
        (B, C), window count (B, 1)) over the earlier windows of each
        recording, or None at its start.
        """
        B, L, C, T = X.shape
        Xf = preprocess(X.reshape(B * L, C, T))
        bp = ((self.band_powers(Xf) - self.bp_mean) / self.bp_std
              ).reshape(B, L, -1)
        if self.use_adapt:
            var = Xf.var(-1).reshape(B, L, C)
            if sums is None:
                sums = (bp.new_zeros(B, bp.shape[-1]), var.new_zeros(B, C),
                        var.new_zeros(B, 1))
            count = sums[2].unsqueeze(1) + torch.arange(
                1, L + 1, device=X.device).float().view(1, L, 1)
            cum_bp = sums[0].unsqueeze(1) + bp.cumsum(1)
            cum_var = sums[1].unsqueeze(1) + var.cumsum(1)
            rel = bp - cum_bp / count
            scale = (cum_var / count).sqrt().clamp_min(
                0.25 * self.chan_std.view(1, 1, C))
            sums = (cum_bp[:, -1], cum_var[:, -1], count[:, -1])
            bp_in = torch.cat([bp, rel], dim=-1)
        else:
            scale = self.chan_std.view(1, 1, C).expand(B, L, C)
            bp_in = bp
        h = self.cnn(Xf / scale.reshape(B * L, C, 1))
        cnn = torch.cat([h.mean(-1), h.std(-1)], dim=1).reshape(B, L, -1)

        idx = torch.arange(L, device=X.device).float().unsqueeze(0)
        idx = idx + torch.as_tensor(start_index, device=X.device,
                                    dtype=torch.float32).reshape(-1, 1)
        t = torch.log1p(idx.expand(B, L) * self.window_s / 60.0).unsqueeze(-1)
        z = self.in_proj(torch.cat([cnn, self.bp_proj(bp_in), t], dim=-1))
        return z, sums

    def forward(self, X, start_index=0, state=None):
        """Sequences (B, L, C, T) of consecutive windows -> logits (B, L, K).

        ``start_index`` is the index of the first window in its recording
        (int or (B,) tensor) and ``state`` = (GRU state, running sums) is
        carried from the previous chunk of the same recordings.
        """
        h, sums = state if state is not None else (None, None)
        z, sums = self.encode(X, start_index, sums)
        out, h = self.gru(z, h)
        return self.head(out), (h, sums)

    # --- streamed inference ---------------------------------------------

    def step_proba(self, X):
        """Next windows (B, C, T) of the current recording -> p (B, K)."""
        # The B windows of a call are consecutive windows of one recording
        # (B = 1 in streamed evaluation).
        logits, self._state = self(X.float().unsqueeze(0), self._n,
                                   self._state)
        self._n += X.shape[0]
        return logits[0].softmax(-1)

    def reset_state(self):
        self._state = None
        self._n = 0


class SleepOnsetEnsemble(nn.Module):
    """Averages the members' time-to-N2 distributions; predicts a quantile."""

    def __init__(self, n_chans, n_times, sfreq, n_members=N_SEEDS,
                 adapt=ADAPT):
        super().__init__()
        self.members = nn.ModuleList(
            SleepOnsetCNNGRU(n_chans, n_times, sfreq, adapt)
            for _ in range(n_members))
        self.register_buffer("quantile", torch.tensor(0.5))

    def _step_proba(self, X):
        return torch.stack([m.step_proba(X) for m in self.members]).mean(0)

    @torch.inference_mode()
    def predict(self, X):
        self.eval()
        p = self._step_proba(X)
        return quantile_seconds(p, self.quantile.item()).clamp(0.0, CAP_S)

    @torch.inference_mode()
    def predict_dist(self, X):
        """(seconds, p over 5 s bins, hazard) for analysis; advances state."""
        self.eval()
        p = self._step_proba(X)
        seconds = quantile_seconds(p, self.quantile.item()).clamp(0.0, CAP_S)
        return seconds, p, p[..., 0]

    def reset_state(self):
        for m in self.members:
            m.reset_state()


# --- training helpers (local only) ------------------------------------------


def _load_windows(dataset):
    """Read every window once, in dataset order, preprocessed, as float16."""
    from torch.utils.data import DataLoader
    Xs, ys, recs, onsets, subs = [], [], [], [], []
    t0 = time.time()
    for X, y, info in DataLoader(dataset, batch_size=512, shuffle=False):
        Xs.append(preprocess(X.float().cpu()).half())
        ys.append(y.float().cpu().reshape(-1))
        recs.append(np.asarray(info["record_id"]).reshape(-1))
        onsets.append(np.asarray(info["onset"], dtype=np.float64).reshape(-1))
        sid = info.get("subject_id", info["record_id"])
        subs.append(np.asarray(sid).reshape(-1))
    print(f"[CNN-GRU] loaded {sum(len(x) for x in Xs)} windows in "
          f"{time.time() - t0:.0f} s")
    return (torch.cat(Xs), torch.cat(ys), np.concatenate(recs),
            np.concatenate(onsets), np.concatenate(subs))


def _recordings(y, rec, onset, sub, rng):
    """Window indices of each recording, in time order, plus its subject.

    Data without real recordings (the Simulated set: one record of random
    windows) is cut into pseudo-recordings sorted by decreasing latency.
    """
    out = []
    if len(np.unique(rec)) < 4:
        perm = rng.permutation(len(y))
        for i, part in enumerate(np.array_split(perm, 10)):
            part = part[np.argsort(-y[part].numpy(), kind="stable")]
            out.append((part, i))
        return out
    for r in np.unique(rec):
        idx = np.flatnonzero(rec == r)
        out.append((idx[np.argsort(onset[idx], kind="stable")],
                    int(sub[idx[0]])))
    return out


def _window_weights(y):
    """Per-window loss weights of one recording, mimicking its W-bMAE.

    Each non-empty metric range gets its severity weight split over its
    windows; the recording's weights sum to 1.
    """
    edges = np.asarray(RANGE_EDGES)
    r = np.clip(np.digitize(y, edges[1:-1]), 0, len(RANGE_WEIGHTS) - 1)
    w = np.zeros(len(y))
    present = np.unique(r)
    total = sum(RANGE_WEIGHTS[b] for b in present)
    for b in present:
        mask = r == b
        w[mask] = RANGE_WEIGHTS[b] / total / mask.sum()
    return torch.as_tensor(w, dtype=torch.float32)


def _soft_labels(y, device):
    """Targets (N,) seconds -> Gaussian-smoothed bin distributions (N, K)."""
    centers = torch.arange(N_BINS, device=device).float()
    pos = (y.clamp(0, CAP_S) / BIN_S).unsqueeze(-1)
    logits = -0.5 * ((centers - pos) / LABEL_SIGMA) ** 2
    return logits.softmax(-1)


def _augment(X):
    """Per-recording amplitude scaling, noise and channel dropout.

    X: (B, L, C, T), preprocessed microvolts.
    """
    B, _, C, _ = X.shape
    scale = torch.empty(B, 1, 1, 1, device=X.device).uniform_(0.7, 1.3)
    X = X * scale + torch.randn_like(X) * 2.0
    drop = torch.rand(B, device=X.device) < 0.1
    if drop.any():
        ch = torch.randint(0, C, (B,), device=X.device)
        mask = torch.ones(B, 1, C, 1, device=X.device)
        mask[drop, 0, ch[drop], 0] = 0.0
        X = X * mask
    return X


def _batch_tensor(X_all, recs, start, stop, device):
    """Pad the windows [start, stop) of each recording into (B, L, C, T)."""
    L = min(stop, max(len(idx) for idx in recs)) - start
    X = torch.zeros(len(recs), L, *X_all.shape[1:], dtype=torch.float16)
    mask = torch.zeros(len(recs), L, dtype=torch.bool)
    for b, idx in enumerate(recs):
        part = idx[start:stop]
        X[b, :len(part)] = X_all[part]
        mask[b, :len(part)] = True
    return X.to(device).float(), mask.to(device)


def _detach(state):
    h, sums = state
    return h.detach(), None if sums is None else tuple(s.detach()
                                                       for s in sums)


@torch.no_grad()
def _sequence_proba(member, X_all, idx, device):
    """Distributions (L, K) of one whole recording, chunked, in eval mode."""
    member.eval()
    out, state = [], None
    for s in range(0, len(idx), 4 * CHUNK):
        X, _ = _batch_tensor(X_all, [idx], s, s + 4 * CHUNK, device)
        logits, state = member(X, s, state)
        out.append(logits[0].softmax(-1))
    return torch.cat(out)


def _scores(probas, y_all, recs):
    """W-bMAE averaged over recordings, for every candidate quantile.

    ``probas[i]`` is the (L, K) distribution of recording ``recs[i]``.
    """
    from benchmark_utils.metrics import binned_mae
    scores = {q: [] for q in QUANTILES}
    for p, (idx, _sub) in zip(probas, recs):
        y = y_all[idx].numpy()
        for q in QUANTILES:
            pred = quantile_seconds(p, q).cpu().numpy()
            scores[q].append(binned_mae(y, pred, RANGE_EDGES, RANGE_WEIGHTS))
    return {q: float(np.mean(s)) for q, s in scores.items()}


def _train_device(default):
    forced = os.environ.get("SLEEP_TRAIN_DEVICE")
    if forced:
        return forced
    if default == "cpu" and torch.backends.mps.is_available():
        return "mps"
    return default


def _train_member(model, seed, X_all, y_all, train_recs, val_recs, weights,
                  device, tag):
    """Train one member in place; keep its best epoch on validation."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=LR,
                            weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)

    best, best_state, bad = float("inf"), None, 0
    for epoch in range(EPOCHS):
        t0 = time.time()
        model.train()
        order = rng.permutation(len(train_recs))
        # Batch recordings of similar length to limit padding.
        order = sorted(order, key=lambda i: len(train_recs[i][0])
                       + rng.uniform(0, 60))
        batches = [order[i:i + RECS_PER_BATCH]
                   for i in range(0, len(order), RECS_PER_BATCH)]
        rng.shuffle(batches)
        running, n_rec = 0.0, 0
        for batch in batches:
            idxs = [train_recs[i][0] for i in batch]
            max_len = max(len(idx) for idx in idxs)
            y = torch.zeros(len(idxs), max_len)
            w = torch.zeros(len(idxs), max_len)
            for b, idx in enumerate(idxs):
                y[b, :len(idx)] = y_all[idx]
                w[b, :len(idx)] = weights[id(idx)]
            y, w = y.to(device), w.to(device)
            state = None
            for s in range(0, max_len, CHUNK):
                X, mask = _batch_tensor(X_all, idxs, s, s + CHUNK, device)
                logits, state = model(_augment(X), s, state)
                state = _detach(state)
                yc, wc = y[:, s:s + CHUNK], w[:, s:s + CHUNK] * mask
                logp = logits.log_softmax(-1)
                ce = -(_soft_labels(yc, device) * logp).sum(-1)
                mean = (logp.exp() * model.bin_values).sum(-1)
                l1 = (mean - yc.clamp(0, CAP_S)).abs() / CAP_S
                loss = ((ce + L1_WEIGHT * l1) * wc).sum() / len(idxs)
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                running += loss.item() * len(idxs)
            n_rec += len(idxs)
        sched.step()

        probas = [_sequence_proba(model, X_all, idx, device)
                  for idx, _ in val_recs]
        q, score = min(_scores(probas, y_all, val_recs).items(),
                       key=lambda kv: kv[1])
        print(f"[CNN-GRU {tag}] epoch {epoch + 1}/{EPOCHS} "
              f"loss {running / n_rec:.3f} val W-bMAE {score:.1f} s "
              f"(q={q}) {time.time() - t0:.0f} s")
        if score < best:
            best, bad = score, 0
            best_state = {k: v.detach().clone()
                          for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    model.eval()
    return best


class Solver(CompetSolver):

    name = "CNN-GRU-Hazard"

    def load_model(self, meta):
        weights = meta["submission_dir"] / WEIGHTS_FILE
        state = None
        # Inference (Codabench): load the shipped weights, which also fix the
        # ensemble size and the adaptation flag. A local training run
        # (train_loader set) starts from scratch instead.
        if weights.exists() and self.train_loader is None:
            state = torch.load(weights, map_location=meta["device"],
                               weights_only=True)
            n_members = len({k.split(".")[1] for k in state
                             if k.startswith("members.")})
            adapt = bool(state["members.0.adapt"])
        else:
            n_members, adapt = N_SEEDS, ADAPT
        model = SleepOnsetEnsemble(meta["n_chans"], meta["n_times"],
                                   float(meta["sfreq"]), n_members, adapt)
        if state is not None:
            model.load_state_dict(state)
        return model.to(meta["device"]).eval()

    def fit(self, model, train_loader):
        rng = np.random.default_rng(SEED)
        X_all, y_all, rec, onset, sub = _load_windows(train_loader.dataset)
        recs = _recordings(y_all, rec, onset, sub, rng)

        # Hold out whole subjects for early stopping and the quantile.
        subjects = np.unique([s for _, s in recs])
        n_val = max(1, int(round(VAL_FRACTION * len(subjects))))
        val_subjects = set(rng.choice(subjects, n_val, replace=False))
        train_recs = [r for r in recs if r[1] not in val_subjects]
        val_recs = [r for r in recs if r[1] in val_subjects]
        print(f"[CNN-GRU] {len(train_recs)} train / {len(val_recs)} val "
              f"recordings ({len(subjects)} subjects); "
              f"{len(model.members)} members, "
              f"adapt={model.members[0].use_adapt}")
        weights = {id(idx): _window_weights(y_all[idx].numpy())
                   for idx, _ in recs}

        # Normalisation stats from the training windows, shared by members.
        train_idx = np.concatenate([idx for idx, _ in train_recs])
        sample = X_all[rng.choice(train_idx, min(20000, len(train_idx)),
                                  replace=False)].float()
        chan_std = sample.std(dim=(0, 2)).clamp_min(1e-3)
        first = model.members[0]
        feats = first.band_powers(sample.to(first.chan_std.device))
        for m in model.members:
            m.chan_std.copy_(chan_std)
            m.bp_mean.copy_(feats.mean(0))
            m.bp_std.copy_(feats.std(0).clamp_min(1e-3))

        out_device = model.quantile.device
        device = _train_device(str(out_device))
        model.to(device)
        print(f"[CNN-GRU] training on {device}")

        member_probas = []
        for i, member in enumerate(model.members):
            best = _train_member(member, SEED + i, X_all, y_all, train_recs,
                                 val_recs, weights, device,
                                 f"{i + 1}/{len(model.members)}")
            print(f"[CNN-GRU] member {i + 1}: best val W-bMAE {best:.1f} s")
            member_probas.append([_sequence_proba(member, X_all, idx, device)
                                  for idx, _ in val_recs])

        ensemble = [torch.stack(ps).mean(0) for ps in zip(*member_probas)]
        q, score = min(_scores(ensemble, y_all, val_recs).items(),
                       key=lambda kv: kv[1])
        model.quantile.fill_(q)
        model.to(out_device).eval()
        print(f"[CNN-GRU] ensemble val W-bMAE {score:.1f} s, q={q:.2f}")
        self._check_streaming(model, X_all, val_recs[0][0], out_device)

    @staticmethod
    @torch.no_grad()
    def _check_streaming(model, X_all, idx, device):
        """Window-by-window predict must match a whole-sequence forward."""
        idx = idx[:200]
        p = torch.stack([_sequence_proba(m, X_all, idx, device)
                         for m in model.members]).mean(0)
        seq = quantile_seconds(p, model.quantile.item())
        model.reset_state()
        stream = torch.cat([model.predict(X_all[i:i + 1].float().to(device))
                            for i in idx])
        model.reset_state()
        diff = (seq.clamp(0, CAP_S) - stream).abs().max().item()
        print(f"[CNN-GRU] streaming check: max |seq - stream| = {diff:.4f} s")
        if diff > BIN_S:
            raise RuntimeError("streamed predictions differ from the "
                               "sequence forward pass")

    def save_model(self, model, path):
        torch.save(model.state_dict(), path / WEIGHTS_FILE)

In [ ]:
%%writefile /content/EEG_sleep_foundation/submission/submission.py
"""Track 03 (Sleep Onset) submission: a basic MLP regressor.

Codabench only calls ``Solver.load_model(meta)`` and the returned model's
``predict(X)``; ``fit`` and ``save_model`` are local-training hooks used by
``benchopt run ... -o "Sleep-onset[training=True]"`` to produce ``weights.pt``.

Contract (see competition/tracks/sleep_onset/README.md):
- input  X: float tensor (B, C, T) on meta["device"], raw microvolts
- output  : (B,) float latencies in seconds to the first N2 epoch, in [0, 600]

This file must stay self-contained: only ``torch`` and packages from the
competition's requirements.txt may be imported, and no other local modules.
"""

import torch
from torch import nn

from benchmark_utils.base_solver import CompetSolver

CAP_S = 600.0
WEIGHTS_FILE = "weights.pt"

HIDDEN = (256, 64)
DROPOUT = 0.2
EPOCHS = 30
LR = 1e-3
WEIGHT_DECAY = 1e-4


class SleepOnsetMLP(nn.Module):
    """(B, C, T) -> per-channel standardisation -> flatten -> MLP -> (B,).

    The per-channel mean/std are fitted on the training set and stored as
    buffers, so they ship inside ``weights.pt`` with the rest of the model.
    """

    def __init__(self, n_chans, n_times, hidden=HIDDEN, dropout=DROPOUT):
        super().__init__()
        self.register_buffer("chan_mean", torch.zeros(1, n_chans, 1))
        self.register_buffer("chan_std", torch.ones(1, n_chans, 1))
        layers, in_dim = [], n_chans * n_times
        for h in hidden:
            layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, X):
        X = (X - self.chan_mean) / self.chan_std
        # Fraction of CAP_S in [0, 1].
        return torch.sigmoid(self.net(X.flatten(1))).squeeze(-1)

    @torch.inference_mode()
    def predict(self, X):
        self.eval()
        return self(X.float()) * CAP_S  # (B,) seconds

    def reset_state(self):
        # Stateless: nothing to reset between streamed recordings.
        pass


class Solver(CompetSolver):

    name = "Sleep-MLP"

    def load_model(self, meta):
        model = SleepOnsetMLP(meta["n_chans"], meta["n_times"])
        weights = meta["submission_dir"] / WEIGHTS_FILE
        # Inference (Codabench): load the shipped weights. A local training
        # run (train_loader set) starts from scratch instead.
        if weights.exists() and self.train_loader is None:
            model.load_state_dict(torch.load(
                weights, map_location=meta["device"], weights_only=True))
        return model.to(meta["device"]).eval()

    def fit(self, model, train_loader):
        # Per-channel normalisation stats over the whole training set.
        total, total_sq, count = 0.0, 0.0, 0
        for X, _y, _info in train_loader:
            X = X.float()
            total = total + X.sum(dim=(0, 2))
            total_sq = total_sq + (X ** 2).sum(dim=(0, 2))
            count += X.shape[0] * X.shape[2]
        mean = total / count
        std = (total_sq / count - mean ** 2).clamp_min(1e-12).sqrt()
        model.chan_mean.copy_(mean.view(1, -1, 1))
        model.chan_std.copy_(std.view(1, -1, 1))

        opt = torch.optim.AdamW(
            model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        # L1 matches the MAE-based track metric.
        loss_fn = nn.L1Loss()
        model.train()
        for epoch in range(EPOCHS):
            running, n = 0.0, 0
            for X, y, _info in train_loader:
                opt.zero_grad()
                loss = loss_fn(model(X.float()), y.float().clamp(0, CAP_S) / CAP_S)
                loss.backward()
                opt.step()
                running += loss.item() * len(y)
                n += len(y)
            print(f"[Sleep-MLP] epoch {epoch + 1}/{EPOCHS} "
                  f"train MAE {running / n * CAP_S:.1f} s")
        model.eval()

    def save_model(self, model, path):
        torch.save(model.state_dict(), path / WEIGHTS_FILE)

In [ ]:
%%writefile /content/EEG_sleep_foundation/scripts/build_submission.py
#!/usr/bin/env python3
"""Train submission/submission.py with Benchopt, re-validate it, and ZIP it.

Steps (mirrors competition/tools/debug_submission.py):
1. Training run (``Sleep-onset[training=True]``): Solver.fit trains the model
   and the export writes submission.py + weights.pt into build/<label>/.
2. Inference-only run on the exported folder, exactly as Codabench runs it,
   to catch architecture/checkpoint mismatches.
3. ZIP the folder's files at the archive root into dist/<label>.zip.

Usage:
    python scripts/build_submission.py                 # synthetic, warm-up shapes
    python scripts/build_submission.py --data muse     # public Muse warm-up data
    python scripts/build_submission.py --data muse --solver submission/my_cnn.py
"""

from __future__ import annotations

import argparse
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
TRACK = ROOT / "competition" / "tracks" / "sleep_onset"
DEFAULT_SOLVER = ROOT / "submission" / "submission.py"

DATASETS = {
    # Synthetic data with the real warm-up tensor contract (4 Muse channels,
    # 5 s at 128 Hz). Validates the pipeline only; the score is meaningless.
    "simulated": "Simulated[n_chans=4,n_times=640,n_train=300,n_test=150,sfreq=128]",
    # Public Muse warm-up data; run `benchopt prepare` on it first.
    "muse": "Interaxon2026Muse",
}


def run(cmd: list[str], env: dict[str, str]) -> None:
    print("+", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True, env=env, cwd=ROOT)


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__,
                                     formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--data", choices=DATASETS, default="simulated")
    parser.add_argument("--solver", type=Path, default=DEFAULT_SOLVER,
                        help="solver file to train (default: submission/submission.py)")
    parser.add_argument("--label",
                        help="output name (default: <solver file stem>_<data>)")
    parser.add_argument("--force", action="store_true",
                        help="overwrite an existing build/dist output")
    args = parser.parse_args()

    benchopt = shutil.which("benchopt") or str(Path(sys.executable).with_name("benchopt"))
    if not Path(benchopt).exists():
        raise SystemExit("benchopt not found; activate the sleep_foundation env")

    solver = args.solver.resolve()
    if not solver.is_file():
        raise SystemExit(f"solver file not found: {solver}")
    stem = "mlp" if solver == DEFAULT_SOLVER else solver.stem
    label = args.label or f"{stem}_{args.data}"
    build_dir = ROOT / "build" / label
    zip_path = ROOT / "dist" / f"{label}.zip"
    if build_dir.exists() or zip_path.exists():
        if not args.force:
            raise SystemExit(f"{build_dir} or {zip_path} exists; use --force")
        shutil.rmtree(build_dir, ignore_errors=True)
        zip_path.unlink(missing_ok=True)
    build_dir.mkdir(parents=True)
    zip_path.parent.mkdir(exist_ok=True)

    env = {**os.environ, "COMPET_SUBMISSION_DIR": str(build_dir)}
    common = [benchopt, "run", str(TRACK), "-d", DATASETS[args.data],
              "-r", "1", "--no-cache", "--no-plot", "--no-html",
              "--no-display", "--local"]

    # 1. Train and export.
    run(common + ["-o", "Sleep-onset[training=True]", "-s", str(solver),
                  "--output", str(ROOT / "build" / f"{label}_train.parquet")], env)
    for name in ("submission.py", "weights.pt"):
        if not (build_dir / name).is_file():
            raise SystemExit(f"training run did not export {name}")

    # 2. Reload the exported folder inference-only, as Codabench does.
    run(common + ["-o", "Sleep-onset[training=False]",
                  "-s", str(build_dir / "submission.py"),
                  "--output", str(ROOT / "build" / f"{label}_eval.parquet")], env)

    # 3. ZIP with every file at the archive root.
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in sorted(build_dir.iterdir()):
            if path.is_file():
                zf.write(path, path.name)
    print(f"\nReady to upload: {zip_path}")
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            print(f"  {info.filename:20s} {info.file_size:>12,d} bytes")
    return 0


if __name__ == "__main__":
    sys.exit(main())

## Install dependencies (keeps Colab's CUDA torch)

In [ ]:
!pip install -q -r competition/requirements.txt

## Muse data: from the Drive cache, else download from NEMAR

In [ ]:
DATA = PROJECT / "competition/tracks/sleep_onset/data"
CACHE = DRIVE_DIR / "muse_data.tar" if USE_DRIVE else None

if not (DATA / "neural_compet").exists() and CACHE is not None and CACHE.exists():
    DATA.mkdir(parents=True, exist_ok=True)
    !tar -xf "{CACHE}" -C "{DATA}"
    print("data restored from Drive")

!benchopt prepare competition/tracks/sleep_onset -d Interaxon2026Muse

if CACHE is not None and not CACHE.exists():
    !tar -cf "{CACHE}" -C "{DATA}" neural_compet
    print("data cached in Drive:", CACHE)

## Smoke test on synthetic data (seconds; the score is meaningless)

In [ ]:
!SLEEP_N_SEEDS=1 python scripts/build_submission.py --solver submission/cnn_gru.py --force 2>&1 | grep -E "CNN-GRU|Error|Ready"

## Train on Muse

Each run trains `Solver.fit` on the GPU, re-runs the exported folder
inference-only (as Codabench does) and writes `dist/<label>.zip`.
`SLEEP_N_SEEDS` = ensemble size, `SLEEP_ADAPT` = per-sleeper adaptation.
Only the `[CNN-GRU ...]` progress lines are shown; the full log goes to
`build/<label>.log`.

In [ ]:
import os, shutil

RUNS = [
    # (label,              n_seeds, adapt)
    ("cnn_gru_adapt",      1, True),
    ("cnn_gru_ens_adapt",  5, True),
    ("cnn_gru_ens",        5, False),
]

def build(label, n_seeds, adapt):
    env = {**os.environ, "SLEEP_N_SEEDS": str(n_seeds),
           "SLEEP_ADAPT": "1" if adapt else "0"}
    log_path = PROJECT / "build" / f"{label}.log"
    cmd = ["python", "scripts/build_submission.py", "--data", "muse",
           "--solver", "submission/cnn_gru.py", "--label", label, "--force"]
    with open(log_path, "w") as log:
        proc = subprocess.Popen(cmd, cwd=PROJECT, env=env, text=True,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in proc.stdout:
            log.write(line)
            if "CNN-GRU" in line or "Error" in line or "Ready" in line:
                print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"{label} failed; see {log_path}")
    if USE_DRIVE:
        out = DRIVE_DIR / "results"
        out.mkdir(exist_ok=True)
        shutil.copytree(PROJECT / "build" / label, out / label, dirs_exist_ok=True)
        for f in (PROJECT / "build").glob(f"{label}*.*"):
            shutil.copy(f, out)
        shutil.copy(PROJECT / "dist" / f"{label}.zip", out)

for label, n_seeds, adapt in RUNS:
    print(f"===== {label}: {n_seeds} seed(s), adapt={adapt}")
    build(label, n_seeds, adapt)

## Results (test W-bMAE on the 40 Muse warm-up recordings)

In [ ]:
import re
import pandas as pd

rows = []
for label, n_seeds, adapt in RUNS:
    eval_file = PROJECT / "build" / f"{label}_eval.parquet"
    if not eval_file.exists():
        continue
    d = pd.read_parquet(eval_file).filter(like="objective_").iloc[-1]
    log = (PROJECT / "build" / f"{label}.log").read_text()
    val = re.findall(r"ensemble val W-bMAE ([\d.]+)", log)
    rows.append({"label": label, "seeds": n_seeds, "adapt": adapt,
                 "val W-bMAE": float(val[-1]) if val else None,
                 "test W-bMAE": d["objective_wbmae_recording_mean"],
                 "bMAE": d["objective_bmae"], "MAE": d["objective_mae"]})
pd.DataFrame(rows).round(1)

Download a submission ZIP to your computer (also in Drive under `results/`):

In [ ]:
from google.colab import files
files.download(str(PROJECT / "dist" / "cnn_gru_ens_adapt.zip"))